# How many eQTLs (SNP-gene pairs) have you identified that are significant?

**TA check:** leave `TA_GRADING = True` and run all cells. Uses only the copied inputs in `checkpoint_data/` and the bundled PLINK executable. Expected: **280,105 tests, 1,317 significant pairs, and 72 eGenes** on chromosome 22.

**Full 1–22 analysis:** set `TA_GRADING = False`. Requires the original project’s full `data/` folder. TA outputs are saved under `checkpoint/results/ta_chr22/`; full outputs remain under `results/project_checkpoint/`.

Week 4 regression, gene-body ±500 kb windows, no PCA adjustment. Chromosome-wide and scan-wide corrections are reported separately.

In [1]:
# TA grading: True = chromosome 22 only; False = chromosomes 1–22.
TA_GRADING = True

from pathlib import Path
import gzip
import json
import re
import subprocess
import time
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

CHECKPOINT = next((candidate
                   for base in (Path.cwd(), *Path.cwd().parents)
                   for candidate in (base, base / "checkpoint")
                   if (candidate / "checkpoint_data/LDREF").is_dir()
                   and (candidate / "tools/plink2").is_file()), None)
if CHECKPOINT is None:
    raise FileNotFoundError("Open this notebook from the checkpoint folder or project root.")
ROOT = CHECKPOINT.parent
DATA = CHECKPOINT / "checkpoint_data" if TA_GRADING else ROOT / "data"
GENOTYPES = DATA / "LDREF"
CHROMOSOMES = (22,) if TA_GRADING else tuple(range(1, 23))
CIS_FLANK_BP = 500_000
ALPHA = 0.05
OUT = CHECKPOINT / "results/ta_chr22" if TA_GRADING else ROOT / "results/project_checkpoint"
WORK = OUT / "work"
WORK.mkdir(parents=True, exist_ok=True)
PLINK = CHECKPOINT / "tools/plink2"
EXPRESSION = (DATA / "expression/GEUVADIS_chr22.tsv.gz" if TA_GRADING else
              DATA / "expression/GD462.GeneQuantRPKM.50FN.samplename.resk10.txt")
GTF = DATA / "annotations/gencode.v12.annotation.gtf.gz"
ANNOT22 = [DATA / "annotations/chr22_GRCh37_genes.json",
           DATA / "annotations/chr22_historical_recovered_genes.json"]
GTF_URL = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_12/gencode.v12.annotation.gtf.gz"
required = [PLINK, EXPRESSION]
if 22 in CHROMOSOMES:
    required += ANNOT22
if any(c != 22 for c in CHROMOSOMES):
    required.append(GTF)
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f"Required input missing: {path}. GTF source: {GTF_URL}")
for chrom in CHROMOSOMES:
    for suffix in ("bed", "bim", "fam"):
        path = GENOTYPES / f"1000G.EUR.{chrom}.{suffix}"
        if not path.is_file():
            raise FileNotFoundError(path)
(OUT / "commands.jsonl").write_text("")

def run_plink(arguments):
    command = [str(PLINK), *map(str, arguments)]
    result = subprocess.run(command, capture_output=True, text=True)
    with (OUT / "commands.jsonl").open("a") as handle:
        handle.write(json.dumps({"command": command, "returncode": result.returncode}) + "\n")
    if result.returncode:
        raise RuntimeError(result.stdout + result.stderr)
    return result.stdout

print(subprocess.check_output([str(PLINK), "--version"], text=True).strip())
print("Output folder:", OUT)


PLINK v2.0.0-b.1-dev M1 (12 Oct 2026)
Output folder: /Users/mary/Documents/School/DSC 180A project/checkpoint/results/ta_chr22


## Match samples and load gene boundaries

Use LDREF and GEUVADIS expression. Gene boundaries are GRCh37: saved Week 4 records for chromosome 22; GENCODE v12 for chromosomes 1–21. The course gene table has different coordinates and is not used for windows. Coordinate differences are saved in `annotation_coordinate_review.tsv`.

In [2]:
expression = pd.read_csv(EXPRESSION, sep="\t")
expression["gene_id"] = expression["TargetID"].str.split(".").str[0]
assert expression.gene_id.is_unique
fam_columns = ["FID", "IID", "father", "mother", "sex", "phenotype"]
fam = pd.read_csv(GENOTYPES / f"1000G.EUR.{CHROMOSOMES[0]}.fam", sep=r"\s+", header=None,
                  names=fam_columns, dtype=str)
assert fam.IID.is_unique
for chrom in CHROMOSOMES:
    other = pd.read_csv(GENOTYPES / f"1000G.EUR.{chrom}.fam", sep=r"\s+",
                        header=None, names=fam_columns, dtype=str)
    assert set(zip(other.FID, other.IID)) == set(zip(fam.FID, fam.IID))
matched = fam.loc[fam.IID.isin(expression.columns), ["FID", "IID"]].copy()
assert len(matched) > 2
matched.to_csv(OUT / "matched_samples.tsv", sep="\t", index=False)
KEEP = WORK / "keep.txt"
matched.to_csv(KEEP, sep="\t", index=False, header=False)
SAMPLE_IDS = matched.IID.tolist()

attributes = re.compile(r'(\w+) "([^"]*)"')
records = []
if any(c != 22 for c in CHROMOSOMES):
    with gzip.open(GTF, "rt") as handle:
        for line in handle:
            if line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) != 9 or f[2] != "gene":
                continue
            chrom = f[0].removeprefix("chr")
            if not chrom.isdigit() or int(chrom) not in CHROMOSOMES or int(chrom) == 22:
                continue
            a = dict(attributes.findall(f[8]))
            records.append({"gene_id": a["gene_id"].split(".")[0], "chrom": int(chrom),
                            "start": int(f[3]), "end": int(f[4]), "symbol": a.get("gene_name", ""),
                            "annotation_source": "GENCODE v12 GRCh37"})
for path in (ANNOT22 if 22 in CHROMOSOMES else []):
    for g in json.loads(path.read_text()):
        assert g["assembly_name"] == "GRCh37"
        if str(g["seq_region_name"]) == "22":
            records.append({"gene_id": g["id"], "chrom": 22, "start": int(g["start"]),
                            "end": int(g["end"]), "symbol": g.get("external_name", g["id"]),
                            "annotation_source": "saved Week 4 GRCh37 records"})
annotation = pd.DataFrame(records)
assert annotation.gene_id.is_unique and (annotation.end >= annotation.start).all()
genes = annotation.merge(expression[["gene_id", "TargetID", "Chr", "Coord"]], on="gene_id", validate="one_to_one")
assert genes.chrom.astype(str).eq(genes.Chr.astype(str)).all()
genes["expression_coord_within_gene"] = genes.Coord.between(genes.start, genes.end)
genes.loc[~genes.expression_coord_within_gene].to_csv(OUT / "annotation_coordinate_review.tsv", sep="\t", index=False)
assert genes.loc[genes.chrom.ne(22), "expression_coord_within_gene"].all()
genes["window_start"] = (genes.start - CIS_FLANK_BP).clip(lower=1)
genes["window_end"] = genes.end + CIS_FLANK_BP
genes = genes.sort_values(["chrom", "start", "gene_id"]).reset_index(drop=True)
genes.to_csv(OUT / "analyzed_gene_annotations.tsv", sep="\t", index=False)
Y_TABLE = expression.set_index("gene_id").loc[:, SAMPLE_IDS].astype(float)
missing = expression.loc[~expression.gene_id.isin(annotation.gene_id), ["TargetID", "gene_id", "Chr", "Coord"]]
missing.to_csv(OUT / "expression_without_autosomal_annotation.tsv", sep="\t", index=False)
print(f"Genotype samples: {len(fam)}; expression samples: {len(expression.columns)-5}; matched: {len(matched)}")
print(f"Measured annotated autosomal genes: {len(genes):,}")



Genotype samples: 489; expression samples: 462; matched: 344
Measured annotated autosomal genes: 633


## Test SNP–gene pairs

Fit `expression ~ SNP dosage`. Use complete observations for each pair; skip pairs with fewer than three observations or no variation. Check a selected pair against `scipy.stats.linregress` on each chromosome.

In [3]:
PAIR_FILE = OUT / "all_cis_pairs.tsv.gz"
columns = ["chrom", "gene_id", "symbol", "snp", "position", "counted_allele", "other_allele", "n", "beta", "se", "p_value"]
gene_summaries = []
total_tests = checked_pairs = 0

def associate(y, X, indices):
    x = X[:, indices]
    ok = np.isfinite(x) & np.isfinite(y[:, None])
    n = ok.sum(axis=0)
    xx = np.where(ok, x, 0); yy = np.where(ok, y[:, None], 0)
    denominator = np.maximum(n, 1)
    sx = xx.sum(axis=0); sy = yy.sum(axis=0)
    sxx = (xx * xx).sum(axis=0) - sx * sx / denominator
    syy = (yy * yy).sum(axis=0) - sy * sy / denominator
    sxy = (xx * yy).sum(axis=0) - sx * sy / denominator
    valid = (n > 2) & (sxx > 1e-12) & (syy > 1e-12)
    ix = indices[valid]; nv = n[valid]
    beta = sxy[valid] / sxx[valid]
    residual = np.maximum(syy[valid] - sxy[valid]**2 / sxx[valid], 0)
    se = np.sqrt(residual / (nv - 2) / sxx[valid])
    with np.errstate(divide="ignore", invalid="ignore"):
        p = 2 * stats.t.sf(np.abs(beta / se), nv - 2)
    good = np.isfinite(beta) & np.isfinite(se) & np.isfinite(p)
    return ix[good], nv[good], beta[good], se[good], p[good]

with gzip.open(PAIR_FILE, "wt") as handle:
    handle.write("\t".join(columns) + "\n")
    for chrom in CHROMOSOMES:
        started = time.monotonic()
        original = GENOTYPES / f"1000G.EUR.{chrom}"
        prefix = WORK / f"full_chr{chrom}"
        run_plink(["--bfile", original, "--keep", KEEP, "--snps-only", "just-acgt", "--export", "A", "--out", prefix])
        raw = pd.read_csv(str(prefix) + ".raw", sep=r"\s+", dtype={"FID": str, "IID": str})
        assert raw.IID.is_unique and set(raw.IID) == set(SAMPLE_IDS)
        raw = raw.set_index("IID").loc[SAMPLE_IDS]
        assert raw.FID.tolist() == matched.FID.tolist()
        bim = pd.read_csv(str(original) + ".bim", sep=r"\s+", header=None,
                          names=["chrom", "snp", "cm", "position", "a1", "a2"], dtype={"snp": str})
        assert bim.snp.is_unique and bim.chrom.eq(chrom).all()
        dosage_columns = list(raw.columns[5:])
        split = [c.rsplit("_", 1) for c in dosage_columns]
        assert [x[0] for x in split] == bim.snp.tolist(), "BIM/export SNP order mismatch."
        bim["counted_allele"] = [x[1] for x in split]
        X = raw.loc[:, dosage_columns].to_numpy(dtype=float)
        assert ((X[np.isfinite(X)] >= 0) & (X[np.isfinite(X)] <= 2)).all()
        # Position sort enables efficient, inclusive cis-window lookup.
        order = np.argsort(bim.position.to_numpy(), kind="stable")
        bim = bim.iloc[order].reset_index(drop=True); X = X[:, order]
        positions = bim.position.to_numpy()
        chrom_tests = 0
        del raw
        for gene in genes.loc[genes.chrom.eq(chrom)].itertuples(index=False):
            y = Y_TABLE.loc[gene.gene_id].to_numpy(dtype=float)
            left = np.searchsorted(positions, gene.window_start, side="left")
            right = np.searchsorted(positions, gene.window_end, side="right")
            ix, n, beta, se, pvals = associate(y, X, np.arange(left, right))
            base = {"chrom": chrom, "gene_id": gene.gene_id, "symbol": gene.symbol,
                    "window_start": gene.window_start, "window_end": gene.window_end}
            if not len(ix):
                gene_summaries.append({**base, "n_tests": 0, "status": "no variable complete-case SNP-expression pair", "lead_snp": "", "lead_p": np.nan})
                continue
            if chrom_tests == 0:
                # Check one pair against scipy on each chromosome.
                k = int(np.argmin(np.abs(pvals - 0.1))); j = ix[k]
                complete = np.isfinite(y) & np.isfinite(X[:, j])
                direct = stats.linregress(X[complete, j], y[complete])
                np.testing.assert_allclose(beta[k], direct.slope, rtol=1e-7, atol=1e-10)
                np.testing.assert_allclose(se[k], direct.stderr, rtol=1e-7, atol=1e-10)
                np.testing.assert_allclose(pvals[k], direct.pvalue, rtol=1e-7, atol=1e-12)
                checked_pairs += 1
            v = bim.iloc[ix]
            other = np.where(v.counted_allele.eq(v.a1), v.a2, v.a1)
            result = pd.DataFrame({"chrom": chrom, "gene_id": gene.gene_id, "symbol": gene.symbol,
                "snp": v.snp.to_numpy(), "position": v.position.to_numpy(),
                "counted_allele": v.counted_allele.to_numpy(), "other_allele": other,
                "n": n, "beta": beta, "se": se, "p_value": pvals})
            result.to_csv(handle, sep="\t", index=False, header=False)
            k = int(np.argmin(pvals))
            gene_summaries.append({**base, "n_tests": len(result), "status": "tested",
                                   "lead_snp": result.snp.iloc[k], "lead_p": pvals[k]})
            chrom_tests += len(result); total_tests += len(result)
        print(f"Chr {chrom:2}: {chrom_tests:,} valid SNP–gene tests ({time.monotonic()-started:.1f}s)")
        del X
summary = pd.DataFrame(gene_summaries)
assert summary.gene_id.is_unique and int(summary.n_tests.sum()) == total_tests and total_tests > 0
summary.to_csv(OUT / "gene_test_summary.tsv", sep="\t", index=False)
print(f"Total: {total_tests:,} tests; {checked_pairs} direct regression checks passed.")


Chr 22: 280,105 valid SNP–gene tests (5.4s)
Total: 280,105 tests; 1 direct regression checks passed.


## Count significant pairs

- **Chromosome-wide:** `p ≤ 0.05 / tests on that chromosome`.
- **Scan-wide:** `p ≤ 0.05 / tests across all selected chromosomes`.

These thresholds are identical in chromosome 22-only mode. Significant pairs are associations, not independent causal effects.

In [4]:
by_chrom = summary.groupby("chrom").agg(genes_considered=("gene_id", "size"), tested_pairs=("n_tests", "sum")).reindex(CHROMOSOMES, fill_value=0)
assert by_chrom.tested_pairs.gt(0).all()
by_chrom["p_threshold_chromosome"] = ALPHA / by_chrom.tested_pairs
scan_threshold = ALPHA / total_tests
by_chrom["p_threshold_scan"] = scan_threshold
chrom_significant = {c: 0 for c in CHROMOSOMES}
scan_significant = {c: 0 for c in CHROMOSOMES}
chrom_genes, scan_genes, scan_snps = set(), set(), set()
rows_read = 0
CHROM_FILE = OUT / "significant_pairs_chromosome_bonferroni.tsv.gz"
SCAN_FILE = OUT / "significant_pairs_scan_bonferroni.tsv.gz"
with gzip.open(CHROM_FILE, "wt") as ch, gzip.open(SCAN_FILE, "wt") as sc:
    header = "\t".join(columns + ["p_bonferroni_chromosome", "p_bonferroni_scan"]) + "\n"
    ch.write(header); sc.write(header)
    for chunk in pd.read_csv(PAIR_FILE, sep="\t", chunksize=100_000):
        assert chunk.p_value.between(0, 1).all()
        rows_read += len(chunk)
        m = chunk.chrom.map(by_chrom.tested_pairs)
        chunk["p_bonferroni_chromosome"] = np.minimum(chunk.p_value * m, 1)
        chunk["p_bonferroni_scan"] = np.minimum(chunk.p_value * total_tests, 1)
        sig_ch = chunk.loc[chunk.p_value.le(ALPHA / m)]
        sig_sc = chunk.loc[chunk.p_value.le(scan_threshold)]
        sig_ch.to_csv(ch, sep="\t", index=False, header=False)
        sig_sc.to_csv(sc, sep="\t", index=False, header=False)
        chrom_genes.update(sig_ch.gene_id)
        scan_genes.update(sig_sc.gene_id)
        scan_snps.update(zip(sig_sc.chrom, sig_sc.snp))
        for c, count in sig_ch.groupby("chrom").size().items(): chrom_significant[int(c)] += int(count)
        for c, count in sig_sc.groupby("chrom").size().items(): scan_significant[int(c)] += int(count)
assert rows_read == total_tests
by_chrom["significant_pairs_chromosome"] = [chrom_significant[c] for c in CHROMOSOMES]
by_chrom["significant_pairs_scan"] = [scan_significant[c] for c in CHROMOSOMES]
by_chrom.to_csv(OUT / "counts_by_chromosome.tsv", sep="\t")
if 22 in CHROMOSOMES:
    assert int(by_chrom.loc[22, "tested_pairs"]) == 280_105, "Week 4 chromosome 22 test count changed; inspect inputs."
    assert chrom_significant[22] == 1_317, "Week 4 chromosome 22 significant-pair count changed; inspect inputs."
chrom_total = sum(chrom_significant.values())
scan_total = sum(scan_significant.values())
summary["lead_p_bonferroni_chromosome"] = np.minimum(summary.lead_p * summary.chrom.map(by_chrom.tested_pairs), 1)
summary["lead_p_bonferroni_scan"] = np.minimum(summary.lead_p * total_tests, 1)
summary.to_csv(OUT / "gene_test_summary.tsv", sep="\t", index=False)
if 22 in CHROMOSOMES:
    print("Chromosome 22 check passed: 280,105 tests; 1,317 significant pairs.")
print(f"Selected chromosomes: {list(CHROMOSOMES)}; {total_tests:,} SNP–gene tests.")
print(f"Chromosome-wide correction: {chrom_total:,} significant pairs (sum of per-chromosome counts).")
print(f"Scan-wide correction: {scan_total:,} significant pairs; raw-p threshold {scan_threshold:.6g}.")
print(f"Scan-wide associated genes: {len(scan_genes):,}; unique SNPs: {len(scan_snps):,}.")
display(by_chrom)

settings = {"chromosomes": list(CHROMOSOMES), "samples": len(matched),
            "model": "Week 4 unadjusted linear regression", "cis_flank_bp": CIS_FLANK_BP,
            "tests": total_tests, "chromosome_corrected_pairs": chrom_total,
            "scan_corrected_pairs": scan_total, "scan_p_threshold": scan_threshold,
            "alpha": ALPHA}
(OUT / "run_settings.json").write_text(json.dumps(settings, indent=2) + "\n")
(OUT / "answer.md").write_text(
    f"# Significant eQTL pairs\n\nChromosomes: {list(CHROMOSOMES)}. "
    f"Samples: {len(matched)}. Tests: {total_tests:,}.\n\n"
    f"- Chromosome-wide Bonferroni: **{chrom_total:,} pairs**.\n"
    f"- Bonferroni across selected chromosomes: **{scan_total:,} pairs** "
    f"(p ≤ {scan_threshold:.6g}).\n\n"
    "Unadjusted Week 4 regression; no PCA correction.\n")
print("Saved results to:", OUT)


Chromosome 22 check passed: 280,105 tests; 1,317 significant pairs.
Selected chromosomes: [22]; 280,105 SNP–gene tests.
Chromosome-wide correction: 1,317 significant pairs (sum of per-chromosome counts).
Scan-wide correction: 1,317 significant pairs; raw-p threshold 1.78504e-07.
Scan-wide associated genes: 72; unique SNPs: 957.
Saved results to: /Users/mary/Documents/School/DSC 180A project/checkpoint/results/ta_chr22


,genes_considered,tested_pairs,p_threshold_chromosome,p_threshold_scan,significant_pairs_chromosome,significant_pairs_scan
chrom,,,,,,
22,633,280105,1.785045e-07,1.785045e-07,1317,1317


# How many eGenes have you identified (genes with at least one eQTL)?

Count each gene once if it has at least one significant SNP association. This follows `TA_GRADING`: **True = chromosome 22 only; False = chromosomes 1–22**. Chromosome 22 should reproduce **72 eGenes out of 633 tested genes**.

In [5]:
def count_egenes(folder):
    gene_results = pd.read_csv(folder / "gene_test_summary.tsv", sep="\t")
    counts = pd.read_csv(folder / "counts_by_chromosome.tsv", sep="\t").set_index("chrom")
    assert gene_results.gene_id.is_unique
    tested = gene_results.n_tests.gt(0)
    gene_results["eGene_chromosome"] = tested & gene_results.lead_p.le(
        ALPHA / gene_results.chrom.map(counts.tested_pairs))
    gene_results["eGene_scan"] = tested & gene_results.lead_p.le(
        ALPHA / counts.tested_pairs.sum())
    table = gene_results.assign(tested=tested).groupby("chrom").agg(
        genes_tested=("tested", "sum"),
        eGenes_chromosome=("eGene_chromosome", "sum"),
        eGenes_scan=("eGene_scan", "sum"))
    if 22 in table.index:
        assert table.loc[22, "genes_tested"] == 633
        assert table.loc[22, "eGenes_chromosome"] == 72
    gene_results.loc[gene_results.eGene_chromosome].to_csv(
        folder / "egenes_chromosome_bonferroni.tsv", sep="\t", index=False)
    gene_results.loc[gene_results.eGene_scan].to_csv(
        folder / "egenes_scan_bonferroni.tsv", sep="\t", index=False)
    table.to_csv(folder / "egenes_by_chromosome.tsv", sep="\t")
    print(f"Chromosomes {list(table.index)}: {int(table.genes_tested.sum()):,} genes tested")
    print(f"Chromosome-wide correction: {int(table.eGenes_chromosome.sum()):,} eGenes")
    print(f"Scan-wide correction: {int(table.eGenes_scan.sum()):,} eGenes")
    display(table)
    return table

print("TA CHECK — CHROMOSOME 22" if TA_GRADING else "FULL ANALYSIS — CHROMOSOMES 1–22")
egenes = count_egenes(OUT)
assert set(egenes.index) == set(CHROMOSOMES)


TA CHECK — CHROMOSOME 22
Chromosomes [22]: 633 genes tested
Chromosome-wide correction: 72 eGenes
Scan-wide correction: 72 eGenes


,genes_tested,eGenes_chromosome,eGenes_scan
chrom,,,
22,633,72,72
